1 ) Lazy event stream processor
Build a lazy event stream system:

In [1]:
import random
from datetime import datetime, timedelta
from itertools import groupby
from collections import Counter, defaultdict

# -----------------------------
# Generate Events
# -----------------------------
random.seed(42)

users = ["Diya", "Raj", "Amit", "Sara", "Kiran"] * 10       # 10 each
event_types = ["login"] * 10 + ["logout"] * 10 + \
              ["purchase"] * 10 + ["error"] * 10 + ["view"] * 10

random.shuffle(users)
random.shuffle(event_types)

base = datetime(2025, 1, 1)

events = []

for i in range(50):
    ts = base + timedelta(
        days=random.randint(0, 30),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )

    events.append({
        "ts": ts,
        "type": event_types[i],
        "user": users[i],
        "data": {"id": i + 1}
    })

# -----------------------------
# 1. Lazy Generator
# -----------------------------
def gen_events():
    for event in events:
        yield event

# -----------------------------
# 2. Filter by Event Type
# -----------------------------
def filter_type(event_stream, event_type):
    return (e for e in event_stream if e["type"] == event_type)

# -----------------------------
# 3. Group by User
# -----------------------------
def group_by_user(event_stream):
    sorted_events = sorted(event_stream, key=lambda e: e["user"])
    return {
        user: list(group)
        for user, group in groupby(sorted_events, key=lambda e: e["user"])
    }

# -----------------------------
# 4. Session Duration
# -----------------------------
def session_duration(event_stream):
    user_events = defaultdict(list)

    for event in sorted(event_stream, key=lambda e: e["ts"]):
        user_events[event["user"]].append(event)

    averages = {}

    for user, evs in user_events.items():
        login = None
        durations = []

        for e in evs:
            if e["type"] == "login":
                login = e["ts"]

            elif e["type"] == "logout" and login:
                durations.append(e["ts"] - login)
                login = None

        if durations:
            avg = sum(
                (d.total_seconds() for d in durations),
                0
            ) / len(durations)

            averages[user] = timedelta(seconds=avg)

    return averages

# -----------------------------
# 5. Event Summary
# -----------------------------
def event_summary():
    all_events = list(gen_events())

    type_counter = Counter(e["type"] for e in all_events)

    user_counter = Counter(e["user"] for e in all_events)
    most_active = user_counter.most_common(1)[0]

    hour_counter = Counter(e["ts"].hour for e in all_events)
    busiest_hour = hour_counter.most_common(1)[0]

    sessions = session_duration(all_events)

    print("=" * 45)
    print("EVENT SUMMARY REPORT")
    print("=" * 45)

    print("\nEvent Summary:")

    for t in ["login", "logout", "purchase", "error", "view"]:
        print(f"{t:<9}: {type_counter[t]}")

    print(f"\nMost active user : {most_active[0]} ({most_active[1]} events)")
    print(f"Busiest hour     : {busiest_hour[0]:02d}:00 ({busiest_hour[1]} events)")

    print("\nSession Durations:")

    if sessions:
        for user, avg in sessions.items():
            total_minutes = int(avg.total_seconds() // 60)
            hrs = total_minutes // 60
            mins = total_minutes % 60
            print(f"  {user:<6}: {hrs}h {mins}m avg")
    else:
        print("  No complete login/logout sessions found.")

# -----------------------------
# Demonstration
# -----------------------------
print("First 5 Events:")
for event in list(gen_events())[:5]:
    print(event)

print("\nPurchase Events:")
for event in filter_type(gen_events(), "purchase"):
    print(event)

print("\nGrouped By User:")
groups = group_by_user(gen_events())
for user, evs in groups.items():
    print(user, "->", len(evs), "events")

print()
event_summary()

First 5 Events:
{'ts': datetime.datetime(2025, 1, 7, 18, 56), 'type': 'view', 'user': 'Diya', 'data': {'id': 1}}
{'ts': datetime.datetime(2025, 1, 23, 10, 13), 'type': 'view', 'user': 'Sara', 'data': {'id': 2}}
{'ts': datetime.datetime(2025, 1, 21, 15, 25), 'type': 'logout', 'user': 'Kiran', 'data': {'id': 3}}
{'ts': datetime.datetime(2025, 1, 29, 20, 29), 'type': 'login', 'user': 'Raj', 'data': {'id': 4}}
{'ts': datetime.datetime(2025, 1, 5, 8, 8), 'type': 'logout', 'user': 'Kiran', 'data': {'id': 5}}

Purchase Events:
{'ts': datetime.datetime(2025, 1, 8, 23, 35), 'type': 'purchase', 'user': 'Diya', 'data': {'id': 6}}
{'ts': datetime.datetime(2025, 1, 16, 2, 48), 'type': 'purchase', 'user': 'Raj', 'data': {'id': 11}}
{'ts': datetime.datetime(2025, 1, 22, 13, 38), 'type': 'purchase', 'user': 'Sara', 'data': {'id': 14}}
{'ts': datetime.datetime(2025, 1, 24, 3, 43), 'type': 'purchase', 'user': 'Diya', 'data': {'id': 19}}
{'ts': datetime.datetime(2025, 1, 4, 9, 27), 'type': 'purchase', 'u

2 ) Combinatorics and DP toolkit
Build a memoized combinatorics toolkit:

In [2]:
import time
from functools import wraps

# ==========================================================
# Memoization Decorator with Cache Statistics
# ==========================================================

def memoize(func):
    cache = {}
    stats = {
        "calls": 0,
        "hits": 0,
        "misses": 0,
        "total_time": 0
    }

    @wraps(func)
    def wrapper(*args):
        stats["calls"] += 1

        if args in cache:
            stats["hits"] += 1
            return cache[args]

        stats["misses"] += 1

        start = time.perf_counter()
        result = func(*args)
        end = time.perf_counter()

        stats["total_time"] += (end - start)
        cache[args] = result

        return result

    wrapper.stats = stats
    return wrapper


# ==========================================================
# 1. Count Paths in Grid with Obstacles
# ==========================================================

def count_paths(grid):
    rows = len(grid)
    cols = len(grid[0])

    @memoize
    def dp(r, c):

        if r >= rows or c >= cols:
            return 0

        if grid[r][c] == "#":
            return 0

        if r == rows - 1 and c == cols - 1:
            return 1

        return dp(r + 1, c) + dp(r, c + 1)

    result = dp(0, 0)
    return result, dp.stats


# ==========================================================
# 2. Word Break
# ==========================================================

def word_break(word, dictionary):

    words = frozenset(dictionary)

    @memoize
    def dp(index):

        if index == len(word):
            return True

        for end in range(index + 1, len(word) + 1):
            if word[index:end] in words and dp(end):
                return True

        return False

    result = dp(0)
    return result, dp.stats


# ==========================================================
# 3. Edit Distance
# ==========================================================

def edit_distance(s1, s2):

    @memoize
    def dp(i, j):

        if i == len(s1):
            return len(s2) - j

        if j == len(s2):
            return len(s1) - i

        if s1[i] == s2[j]:
            return dp(i + 1, j + 1)

        return 1 + min(
            dp(i + 1, j),      # Delete
            dp(i, j + 1),      # Insert
            dp(i + 1, j + 1)   # Replace
        )

    result = dp(0, 0)
    return result, dp.stats


# ==========================================================
# 4. Coin Combinations
# ==========================================================

def coin_combinations(coins, amount):

    coins = tuple(coins)

    @memoize
    def dp(index, remaining):

        if remaining == 0:
            return 1

        if remaining < 0:
            return 0

        if index == len(coins):
            return 0

        return (
            dp(index, remaining - coins[index]) +
            dp(index + 1, remaining)
        )

    result = dp(0, amount)
    return result, dp.stats


# ==========================================================
# Cache Statistics Printer
# ==========================================================

def print_stats(stats):
    hits = stats["hits"]
    misses = stats["misses"]
    calls = stats["calls"]

    hit_rate = (hits / calls * 100) if calls else 0
    avg_time = (stats["total_time"] / misses * 1000) if misses else 0

    print(f"Total Calls        : {calls}")
    print(f"Cache Hits         : {hits}")
    print(f"Cache Misses       : {misses}")
    print(f"Hit Rate           : {hit_rate:.2f}%")
    print(f"Avg Compute Time   : {avg_time:.6f} ms")


# ==========================================================
# Testing
# ==========================================================

grid = [
    ['.', '.', '.', '.'],
    ['.', '#', '.', '.'],
    ['.', '.', '#', '.'],
    ['.', '.', '.', '.']
]

print("=" * 60)
print("1. Grid Paths")
print("=" * 60)

start = time.perf_counter()
paths, stats = count_paths(grid)
end = time.perf_counter()

print("Paths in grid:", paths)
print(f"Execution Time: {(end-start)*1000:.4f} ms")
print_stats(stats)


print("\n" + "=" * 60)
print("2. Word Break")
print("=" * 60)

start = time.perf_counter()
ans1, stats = word_break("leetcode", ["leet", "code"])
ans2, _ = word_break("applepenapple", ["apple", "pen"])
end = time.perf_counter()

print("word_break leetcode:", ans1)
print("word_break applepenapple:", ans2)
print(f"Execution Time: {(end-start)*1000:.4f} ms")
print_stats(stats)


print("\n" + "=" * 60)
print("3. Edit Distance")
print("=" * 60)

start = time.perf_counter()
dist, stats = edit_distance("kitten", "sitting")
end = time.perf_counter()

print("Edit distance kitten → sitting:", dist)
print(f"Execution Time: {(end-start)*1000:.4f} ms")
print_stats(stats)


print("\n" + "=" * 60)
print("4. Coin Combinations")
print("=" * 60)

start = time.perf_counter()
ways, stats = coin_combinations([1, 2, 5], 5)
end = time.perf_counter()

print("Coin combinations for 5:", ways)
print(f"Execution Time: {(end-start)*1000:.4f} ms")
print_stats(stats)

1. Grid Paths
Paths in grid: 4
Execution Time: 0.1226 ms
Total Calls        : 27
Cache Hits         : 5
Cache Misses       : 22
Hit Rate           : 18.52%
Avg Compute Time   : 0.006741 ms

2. Word Break
word_break leetcode: True
word_break applepenapple: True
Execution Time: 0.2223 ms
Total Calls        : 3
Cache Hits         : 0
Cache Misses       : 3
Hit Rate           : 0.00%
Avg Compute Time   : 0.009933 ms

3. Edit Distance
Edit distance kitten → sitting: 3
Execution Time: 0.2262 ms
Total Calls        : 113
Cache Hits         : 57
Cache Misses       : 56
Hit Rate           : 50.44%
Avg Compute Time   : 0.013043 ms

4. Coin Combinations
Coin combinations for 5: 4
Execution Time: 0.1495 ms
Total Calls        : 31
Cache Hits         : 3
Cache Misses       : 28
Hit Rate           : 9.68%
Avg Compute Time   : 0.006786 ms


3 ) Functional validation framework
Build a complete data validation framework using HOFs:

In [4]:
import re
import inspect
from functools import wraps

# ==========================================================
# Validator Factory Functions
# ==========================================================

def required():
    def validator(value):
        if value is None or value == "":
            return False, "required"
        return True, ""
    return validator


def type_check(*types):
    def validator(value):
        if not isinstance(value, types):
            return False, "wrong type"
        return True, ""
    return validator


def range_check(min_value, max_value):
    def validator(value):
        if not isinstance(value, (int, float)):
            return False, "wrong type"

        if not (min_value <= value <= max_value):
            return False, "out of range"

        return True, ""
    return validator


def length_check(min_len, max_len):
    def validator(value):
        if not (min_len <= len(value) <= max_len):
            return False, "too short"
        return True, ""
    return validator


def pattern_check(regex):
    pattern = re.compile(regex)

    def validator(value):
        if not pattern.fullmatch(str(value)):
            return False, "invalid pattern"
        return True, ""
    return validator


# ==========================================================
# Combine Validators
# ==========================================================

def combine(*validators, mode="all"):

    def validator(value):
        errors = []

        if mode == "all":
            for v in validators:
                ok, msg = v(value)
                if not ok:
                    errors.append(msg)
            return len(errors) == 0, errors

        elif mode == "any":
            for v in validators:
                ok, msg = v(value)
                if ok:
                    return True, []
                errors.append(msg)
            return False, errors

    return validator


# ==========================================================
# Validate Record
# ==========================================================

def validate_record(record, schema):

    errors = {}

    for field, validators in schema.items():

        value = record.get(field)
        field_errors = []

        for validator in validators:
            try:
                ok, msg = validator(value)
            except TypeError:
                field_errors.append("wrong type")
                break

            if not ok:
                if isinstance(msg, list):
                    field_errors.extend(msg)
                else:
                    field_errors.append(msg)

                # Stop further validation if type is wrong
                if msg == "wrong type":
                    break

        if field_errors:
            errors[field] = field_errors

    return len(errors) == 0, errors

# ==========================================================
# Decorator
# ==========================================================

def validated(schema):

    def decorator(func):

        @wraps(func)
        def wrapper(*args, **kwargs):

            bound = inspect.signature(func).bind(*args, **kwargs)
            bound.apply_defaults()

            valid, errors = validate_record(bound.arguments, schema)

            if not valid:
                print("@validated decorator: blocked invalid call")
                print(errors)
                return

            return func(*args, **kwargs)

        return wrapper

    return decorator


# ==========================================================
# Student Registration Schema
# ==========================================================

student_schema = {

    "name": [
        required(),
        type_check(str),
        length_check(3, 30),
        pattern_check(r"[A-Za-z ]+")
    ],

    "age": [
        required(),
        type_check(int),
        range_check(18, 30)
    ],

    "email": [
        required(),
        type_check(str),
        pattern_check(r"^[\w\.-]+@[\w\.-]+\.\w+$")
    ],

    "score": [
        required(),
        type_check(int, float),
        range_check(0, 100)
    ]
}


# ==========================================================
# Test Records
# ==========================================================

valid_student = {
    "name": "Diya Desai",
    "age": 21,
    "email": "diya@gmail.com",
    "score": 92
}

invalid_student = {
    "name": "D1",
    "age": 40,
    "email": "abc.com",
    "score": "A+"
}


print("=" * 50)
print("VALID RECORD")
print("=" * 50)

valid, errors = validate_record(valid_student, student_schema)
print("Valid record:", valid)
print(errors)

print("\n" + "=" * 50)
print("INVALID RECORD")
print("=" * 50)

valid, errors = validate_record(invalid_student, student_schema)

print("Valid record:", valid)

for field, msgs in errors.items():
    print(f"{field}: {msgs}")


# ==========================================================
# Decorator Test
# ==========================================================

@validated(student_schema)
def register_student(name, age, email, score):
    print(f"Registered: {name}, age {age}")


print("\n" + "=" * 50)
print("DECORATOR TEST")
print("=" * 50)

# Invalid call
register_student("D1", 40, "abc.com", "A+")

# Valid call
register_student("Diya Desai", 21, "diya@gmail.com", 95)

VALID RECORD
Valid record: True
{}

INVALID RECORD
Valid record: False
name: ['too short', 'invalid pattern']
age: ['out of range']
email: ['invalid pattern']
score: ['wrong type']

DECORATOR TEST
@validated decorator: blocked invalid call
{'name': ['too short', 'invalid pattern'], 'age': ['out of range'], 'email': ['invalid pattern'], 'score': ['wrong type']}
Registered: Diya Desai, age 21


4) File system report generator

In [5]:
import random
from datetime import datetime, timedelta
from collections import defaultdict
import os

# ==========================================================
# Custom Exception
# ==========================================================

class FileSystemError(Exception):
    pass

# ==========================================================
# Virtual File System
# ==========================================================

fs = {
    "projects/": [
        "app.py",
        "utils.py",
        "test_app.py",
        "config.json"
    ],

    "projects/data/": [
        "sales.csv",
        "users.csv",
        "products.json"
    ],

    "projects/logs/": [
        "app.log",
        "error.log",
        "access.log"
    ],

    "projects/backups/": [
        "backup_2025-01-01.zip",
        "backup_2025-01-15.zip"
    ]
}

# ==========================================================
# Generate Metadata
# ==========================================================

random.seed(42)

permissions = ["r", "rw", "rwx"]

metadata = {}

today = datetime.now()

for folder, files in fs.items():
    for file in files:

        metadata[folder + file] = {
            "size": random.randint(1024, 50 * 1024 * 1024),
            "modified": today - timedelta(days=random.randint(0, 30)),
            "permission": random.choice(permissions)
        }

# Make duplicate sizes intentionally
paths = list(metadata.keys())
metadata[paths[1]]["size"] = metadata[paths[0]]["size"]
metadata[paths[5]]["size"] = metadata[paths[4]]["size"]

# ==========================================================
# Human Readable Size
# ==========================================================

def format_size(size):

    units = ["B", "KB", "MB", "GB"]

    for unit in units:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024

# ==========================================================
# Generator
# ==========================================================

def file_gen(fs):

    for folder, files in fs.items():

        for file in files:

            path = folder + file

            if metadata[path]["permission"] == "":
                raise FileSystemError(f"Access denied: {path}")

            yield {
                "path": path,
                "name": file,
                **metadata[path]
            }

# ==========================================================
# Size Report
# ==========================================================

def size_report():

    total = 0
    ext_sizes = defaultdict(int)
    ext_counts = defaultdict(int)

    largest = None

    for f in file_gen(fs):

        total += f["size"]

        ext = os.path.splitext(f["name"])[1]

        ext_sizes[ext] += f["size"]
        ext_counts[ext] += 1

        if largest is None or f["size"] > largest["size"]:
            largest = f

    print("=" * 50)
    print("FILE SYSTEM REPORT")
    print("=" * 50)

    print("Total files:", len(metadata))
    print("Total size :", format_size(total))

    print("\nBy Extension:")

    for ext in sorted(ext_sizes):
        print(
            f"{ext:<6}: {ext_counts[ext]:2d} files | "
            f"{format_size(ext_sizes[ext])}"
        )

    print(f"\nLargest: {largest['name']} ({format_size(largest['size'])})")

# ==========================================================
# Recent Files
# ==========================================================

def recent_files(days=7):

    cutoff = today - timedelta(days=days)

    recent = [
        f for f in file_gen(fs)
        if f["modified"] >= cutoff
    ]

    print(f"\nRecent ({days} days): {len(recent)} files")

    for f in recent:
        print(" ", f["name"])

# ==========================================================
# Duplicate Finder
# ==========================================================

def duplicate_finder():

    groups = defaultdict(list)

    for f in file_gen(fs):
        groups[f["size"]].append(f["name"])

    duplicates = {
        size: files
        for size, files in groups.items()
        if len(files) > 1
    }

    print("\nPotential duplicates:", len(duplicates), "pairs")

    for size, files in duplicates.items():
        print(format_size(size), "->", files)

# ==========================================================
# Directory Tree
# ==========================================================

def print_tree():

    print("\nDirectory Tree:\n")

    for folder in sorted(fs):

        level = folder.count("/") - 1

        indent = "│   " * level

        print(indent + folder.split("/")[-2] + "/")

        for file in fs[folder]:

            path = folder + file

            print(
                indent +
                "├── " +
                file +
                f" ({format_size(metadata[path]['size'])})"
            )

# ==========================================================
# Main
# ==========================================================

try:

    size_report()

    recent_files(7)

    duplicate_finder()

    print_tree()

except FileSystemError as e:
    print("FileSystemError:", e)

FILE SYSTEM REPORT
Total files: 12
Total size : 352.92 MB

By Extension:
.csv  :  2 files | 69.81 MB
.json :  2 files | 12.56 MB
.log  :  3 files | 109.86 MB
.py   :  3 files | 96.14 MB
.zip  :  2 files | 64.56 MB

Largest: access.log (41.59 MB)

Recent (7 days): 7 files
  app.py
  test_app.py
  sales.csv
  users.csv
  products.json
  error.log
  backup_2025-01-01.zip

Potential duplicates: 2 pairs
40.92 MB -> ['app.py', 'utils.py']
34.90 MB -> ['sales.csv', 'users.csv']

Directory Tree:

projects/
├── app.py (40.92 MB)
├── utils.py (40.92 MB)
├── test_app.py (14.29 MB)
├── config.json (6.56 MB)
│   backups/
│   ├── backup_2025-01-01.zip (26.85 MB)
│   ├── backup_2025-01-15.zip (37.71 MB)
│   data/
│   ├── sales.csv (34.90 MB)
│   ├── users.csv (34.90 MB)
│   ├── products.json (6.00 MB)
│   logs/
│   ├── app.log (32.34 MB)
│   ├── error.log (35.92 MB)
│   ├── access.log (41.59 MB)


5) Personal Finance Tracker

In [7]:
import random
import math
from datetime import datetime, timedelta
from functools import reduce, partial, lru_cache
from collections import Counter, defaultdict, namedtuple

# ==========================================================
# Custom Exception
# ==========================================================

class FinanceError(Exception):
    pass

# ==========================================================
# Transaction Structure
# ==========================================================

Transaction = namedtuple(
    "Transaction",
    ["date", "category", "amount", "description"]
)

# ==========================================================
# Constants
# ==========================================================

random.seed(42)

categories = [
    "Food",
    "Transport",
    "Entertainment",
    "Bills",
    "Shopping",
    "Income"
]

descriptions = {
    "Food": [
        "Restaurant",
        "Groceries",
        "Cafe",
        "Snacks"
    ],
    "Transport": [
        "Bus",
        "Taxi",
        "Fuel",
        "Train"
    ],
    "Entertainment": [
        "Movie",
        "Games",
        "Netflix",
        "Concert"
    ],
    "Bills": [
        "Electricity",
        "Internet",
        "Water",
        "Mobile"
    ],
    "Shopping": [
        "Clothes",
        "Amazon",
        "Electronics",
        "Shoes"
    ],
    "Income": [
        "Salary",
        "Freelance",
        "Bonus"
    ]
}

# ==========================================================
# Generate 60 Transactions (Jan-Jun 2025)
# ==========================================================

transactions = []

start_date = datetime(2025, 1, 1)

for month in range(6):

    month_start = datetime(2025, month + 1, 1)

    # Income every month
    income = random.randint(35000, 45000)

    transactions.append(
        Transaction(
            month_start,
            "Income",
            income,
            "Salary"
        )
    )

    # Remaining expenses
    for _ in range(9):

        category = random.choice(categories[:-1])

        amount = -random.randint(300, 6000)

        day = random.randint(1, 28)

        date = datetime(2025, month + 1, day)

        desc = random.choice(descriptions[category])

        transactions.append(
            Transaction(
                date,
                category,
                amount,
                desc
            )
        )

# Sort by date
transactions.sort(key=lambda x: x.date)

# ==========================================================
# Generator
# ==========================================================

def transaction_stream():

    for t in transactions:
        yield t

# ==========================================================
# Helper Functions
# ==========================================================

def rupees(amount):
    return f"Rs.{amount:,.2f}"

month_names = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun"
}

# ==========================================================
# functools.partial Example
# ==========================================================

food_filter = partial(
    filter,
    lambda t: t.category == "Food"
)

# ==========================================================
# lru_cache Example
# ==========================================================

@lru_cache(maxsize=None)
def month_name(num):
    return month_names[num]

# ==========================================================
# HOF Examples
# ==========================================================

expense_transactions = list(
    filter(
        lambda t: t.amount < 0,
        transaction_stream()
    )
)

expense_amounts = list(
    map(
        lambda t: abs(t.amount),
        expense_transactions
    )
)

# ==========================================================
# Comprehension Examples
# ==========================================================

category_colors = {
    c: random.choice(
        ["Red", "Blue", "Green", "Yellow"]
    )
    for c in categories
}

month_numbers = [
    t.date.month
    for t in transaction_stream()
]

# ==========================================================
# zip + enumerate Example
# ==========================================================

for i, (m, c) in enumerate(
    zip(month_names.values(), category_colors.keys()),
    start=1
):
    pass

print("Generated Transactions:", len(transactions))
print("Generator Ready.")

Generated Transactions: 60
Generator Ready.


In [9]:
# ==========================================================
# Monthly Income & Expenses
# ==========================================================

def monthly_summary():

    summary = defaultdict(lambda: {"income": 0, "expense": 0})

    for t in transaction_stream():

        month = t.date.month

        if t.amount > 0:
            summary[month]["income"] += t.amount
        else:
            summary[month]["expense"] += abs(t.amount)

    return summary


# ==========================================================
# Category Breakdown
# ==========================================================

def category_breakdown():

    category_total = defaultdict(int)

    for t in transaction_stream():

        if t.amount < 0:
            category_total[t.category] += abs(t.amount)

    total = sum(category_total.values())

    result = {}

    for cat, amt in category_total.items():

        percent = (amt / total * 100) if total else 0

        result[cat] = (amt, percent)

    return result


# ==========================================================
# Savings Rate Per Month
# ==========================================================

def savings_rate():

    summary = monthly_summary()

    result = {}

    for month, values in summary.items():

        income = values["income"]
        expense = values["expense"]

        saved = income - expense

        rate = (saved / income * 100) if income else 0

        result[month] = (income, expense, saved, rate)

    return result


# ==========================================================
# Predict Next Month Spending
# ==========================================================

def predict_next_month():

    summary = monthly_summary()

    expenses = [
        summary[m]["expense"]
        for m in sorted(summary)
    ]

    if len(expenses) < 3:
        return 0

    return sum(expenses[-3:]) / 3


# ==========================================================
# Spending Anomalies
# ==========================================================

def find_anomalies():

    expenses = [
        abs(t.amount)
        for t in transaction_stream()
        if t.amount < 0
    ]

    mean = sum(expenses) / len(expenses)

    variance = sum(
        (x - mean) ** 2
        for x in expenses
    ) / len(expenses)

    std = math.sqrt(variance)

    anomalies = []

    for t in transaction_stream():

        if t.amount < 0:

            if abs(t.amount) > mean + 2 * std:
                anomalies.append(t)

    return anomalies


# ==========================================================
# Recursive Spending Tree
# ==========================================================

def spending_tree(categories, index=0):

    if index >= len(categories):
        return 0

    current = categories[index]

    amount = sum(
        abs(t.amount)
        for t in transaction_stream()
        if t.category == current
    )

    print(f"{'  '*index}{current:<15} {rupees(amount)}")

    return amount + spending_tree(categories, index + 1)


# ==========================================================
# Overall Savings Rate
# ==========================================================

def overall_savings():

    summary = monthly_summary()

    total_income = reduce(
        lambda x, y: x + y,
        [v["income"] for v in summary.values()]
    )

    total_expense = reduce(
        lambda x, y: x + y,
        [v["expense"] for v in summary.values()]
    )

    saved = total_income - total_expense

    rate = (saved / total_income * 100) if total_income else 0

    return total_income, total_expense, saved, rate


# ==========================================================
# Category Counter
# ==========================================================

def category_counter():

    return Counter(
        t.category
        for t in transaction_stream()
    )


# ==========================================================
# Transactions By Month
# ==========================================================

def transactions_by_month():

    months = defaultdict(list)

    for t in transaction_stream():
        months[t.date.month].append(t)

    return months


# ==========================================================
# Largest Expense
# ==========================================================

def largest_expense():

    expenses = list(
        filter(
            lambda t: t.amount < 0,
            transaction_stream()
        )
    )

    return max(
        expenses,
        key=lambda t: abs(t.amount)
    )


# ==========================================================
# Average Expense
# ==========================================================

def average_expense():

    expenses = list(
        map(
            lambda t: abs(t.amount),
            filter(
                lambda t: t.amount < 0,
                transaction_stream()
            )
        )
    )

    return sum(expenses) / len(expenses)


# ==========================================================
# Highest Income
# ==========================================================

def highest_income():

    incomes = [
        t
        for t in transaction_stream()
        if t.amount > 0
    ]

    return max(
        incomes,
        key=lambda t: t.amount
    )

# ==========================================================
# Print Complete Financial Report
# ==========================================================

def print_report():

    print("=" * 60)
    print("        PERSONAL FINANCE REPORT")
    print("=" * 60)
    print("Period : Jan 2025 - Jun 2025")

    # ---------------- Monthly Summary ----------------
    print("\nMonthly Summary:")

    summary = savings_rate()

    for month in sorted(summary):

        income, expense, saved, rate = summary[month]

        print(
            f"{month_name(month):<3} | "
            f"Income: {rupees(income):>12} | "
            f"Expenses: {rupees(expense):>12} | "
            f"Saved: {rupees(saved):>12} "
            f"({rate:5.1f}%)"
        )

    # ---------------- Category Breakdown ----------------
    print("\nCategory Breakdown:")

    breakdown = category_breakdown()

    for cat, (amt, percent) in breakdown.items():

        print(
            f"{cat:<15}: "
            f"{rupees(amt):>12} "
            f"({percent:5.1f}%)"
        )

    # ---------------- Counter ----------------
    print("\nTransaction Count:")

    counter = category_counter()

    for cat, count in counter.items():
        print(f"{cat:<15}: {count}")

    # ---------------- Largest Expense ----------------
    largest = largest_expense()

    print("\nLargest Expense:")
    print(
        f"{largest.category} | "
        f"{rupees(abs(largest.amount))} | "
        f"{largest.description}"
    )

    # ---------------- Highest Income ----------------
    highest = highest_income()

    print("\nHighest Income:")
    print(
        f"{rupees(highest.amount)} "
        f"on {highest.date.strftime('%d-%b-%Y')}"
    )

    # ---------------- Average Expense ----------------
    print(
        f"\nAverage Expense : "
        f"{rupees(average_expense())}"
    )

    # ---------------- Prediction ----------------
    predicted = predict_next_month()

    print(
        f"Predicted Next Month Spending : "
        f"{rupees(predicted)}"
    )

    # ---------------- Anomalies ----------------
    anomalies = find_anomalies()

    print(
        f"\nAnomalies detected : "
        f"{len(anomalies)}"
    )

    for t in anomalies:

        print(
            f"  {t.date.strftime('%d-%b')} | "
            f"{t.category:<15} | "
            f"{rupees(abs(t.amount))}"
        )

    # ---------------- Spending Tree ----------------
    print("\nRecursive Spending Tree")

    spending_tree([
        "Food",
        "Transport",
        "Entertainment",
        "Bills",
        "Shopping"
    ])

    # ---------------- Overall Savings ----------------
    income, expense, saved, rate = overall_savings()

    print("\nOverall Summary")

    print(f"Total Income   : {rupees(income)}")
    print(f"Total Expenses : {rupees(expense)}")
    print(f"Total Saved    : {rupees(saved)}")
    print(f"Savings Rate   : {rate:.2f}%")

    print("=" * 60)


# ==========================================================
# Exception Handling
# ==========================================================

try:

    if len(transactions) != 60:
        raise FinanceError("Expected exactly 60 transactions.")

    print_report()

except FinanceError as e:

    print("Finance Error:", e)

except Exception as e:

    print("Unexpected Error:", e)
    

        PERSONAL FINANCE REPORT
Period : Jan 2025 - Jun 2025

Monthly Summary:
Jan | Income: Rs.36,824.00 | Expenses: Rs.20,766.00 | Saved: Rs.16,058.00 ( 43.6%)
Feb | Income: Rs.36,584.00 | Expenses: Rs.23,850.00 | Saved: Rs.12,734.00 ( 34.8%)
Mar | Income: Rs.44,980.00 | Expenses: Rs.33,605.00 | Saved: Rs.11,375.00 ( 25.3%)
Apr | Income: Rs.44,577.00 | Expenses: Rs.23,175.00 | Saved: Rs.21,402.00 ( 48.0%)
May | Income: Rs.42,433.00 | Expenses: Rs.23,152.00 | Saved: Rs.19,281.00 ( 45.4%)
Jun | Income: Rs.44,007.00 | Expenses: Rs.27,713.00 | Saved: Rs.16,294.00 ( 37.0%)

Category Breakdown:
Shopping       : Rs.31,726.00 ( 20.8%)
Entertainment  : Rs.19,072.00 ( 12.5%)
Food           : Rs.37,272.00 ( 24.5%)
Transport      : Rs.52,157.00 ( 34.3%)
Bills          : Rs.12,034.00 (  7.9%)

Transaction Count:
Income         : 6
Shopping       : 13
Entertainment  : 9
Food           : 14
Transport      : 14
Bills          : 4

Largest Expense:
Transport | Rs.5,790.00 | Bus

Highest Income:
Rs.44